In [4]:
# Install TensorFlow with GPU support for Python 3.10.15
# Your system: NVIDIA RTX 4080 with CUDA 13.0 driver support

# Step 1: Install TensorFlow with CUDA support (bundles CUDA libraries)
# tensorflow[and-cuda] includes all necessary CUDA/cuDNN libraries
print("Installing TensorFlow with GPU support...")
%pip install tensorflow[and-cuda]>=2.15.0 scikit-learn>=1.2.2 seaborn>=0.13.2 opencv-python>=4.11.0.86 matplotlib>=3.10.0 numpy>=1.26.4 pandas>=2.2.2 -q

print("\n✅ Installation complete!")
print("⚠️  IMPORTANT: RESTART THE KERNEL after this cell completes")
print("   Then run Cell 1 to verify GPU detection")

Installing TensorFlow with GPU support...
zsh:1: no matches found: tensorflow[and-cuda]
Note: you may need to restart the kernel to use updated packages.

✅ Installation complete!
⚠️  IMPORTANT: RESTART THE KERNEL after this cell completes
   Then run Cell 1 to verify GPU detection


In [1]:
# GPU Setup and Verification
# This cell verifies GPU detection and installs CUDA libraries if needed

import subprocess
import sys
import os

print("=" * 60)
print("GPU SETUP AND VERIFICATION")
print("=" * 60)

# Check if TensorFlow is installed
try:
    import tensorflow as tf
    print(f"✅ TensorFlow version: {tf.__version__}")
except ImportError:
    print("⚠️  TensorFlow not found. Installing...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tensorflow[and-cuda]>=2.15.0', '-q'])
    import tensorflow as tf
    print(f"✅ TensorFlow installed: {tf.__version__}")

# Check for CUDA libraries
print("\nChecking CUDA libraries...")
cuda_packages = ['nvidia-cudnn-cu12', 'nvidia-cublas-cu12', 'nvidia-cuda-nvrtc-cu12']
missing_packages = []

for package in cuda_packages:
    result = subprocess.run([sys.executable, '-m', 'pip', 'show', package], 
                           capture_output=True, text=True)
    if result.returncode != 0:
        missing_packages.append(package)
    else:
        print(f"  ✅ {package} installed")

if missing_packages:
    print(f"\n⚠️  Missing CUDA packages: {missing_packages}")
    print("Installing tensorflow[and-cuda] to get all CUDA libraries...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tensorflow[and-cuda]>=2.15.0', '-q', '--upgrade'])
    print("✅ CUDA libraries installed!")
    print("⚠️  RESTART KERNEL NOW, then re-run this cell to verify GPU")
else:
    print("\n✅ All CUDA libraries are installed")
    
print("\n" + "=" * 60)
print("Next: Run Cell 2 to verify GPU detection")
print("=" * 60)


GPU SETUP AND VERIFICATION


2025-12-09 09:45:10.958108: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-09 09:45:11.664111: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-09 09:45:13.748048: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


✅ TensorFlow version: 2.20.0

Checking CUDA libraries...
  ✅ nvidia-cudnn-cu12 installed
  ✅ nvidia-cublas-cu12 installed
  ✅ nvidia-cuda-nvrtc-cu12 installed

✅ All CUDA libraries are installed

Next: Run Cell 2 to verify GPU detection


In [2]:
# Verify GPU Detection
import tensorflow as tf

print("=" * 60)
print("GPU DETECTION VERIFICATION")
print("=" * 60)

# List all devices
devices = tf.config.list_physical_devices()
print(f"\nTensorFlow version: {tf.__version__}")
print(f"\nAll available devices:")
for device in devices:
    print(f"  - {device}")

# Check specifically for GPU
gpus = tf.config.list_physical_devices('GPU')
print(f"\nGPU devices found: {len(gpus)}")

if len(gpus) > 0:
    print("\n✅ GPU DETECTED!")
    for i, gpu in enumerate(gpus):
        print(f"  GPU {i}: {gpu.name}")
        try:
            gpu_details = tf.config.experimental.get_device_details(gpu)
            print(f"    Details: {gpu_details}")
        except:
            pass
else:
    print("\n⚠️  NO GPU DETECTED")
    print("\nTroubleshooting steps:")
    print("  1. Make sure you RESTARTED the kernel after installing TensorFlow")
    print("  2. Verify NVIDIA driver: Run 'nvidia-smi' in terminal")
    print("  3. Check CUDA libraries are installed")
    print("  4. For WSL2: Ensure GPU passthrough is enabled")
    
print("\n" + "=" * 60)

GPU DETECTION VERIFICATION

TensorFlow version: 2.20.0

All available devices:
  - PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')
  - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

GPU devices found: 1

✅ GPU DETECTED!
  GPU 0: /physical_device:GPU:0
    Details: {'compute_capability': (8, 9), 'device_name': 'NVIDIA GeForce RTX 4080'}



In [3]:
# Import libraries and configure GPU
from tensorflow.keras import losses
from tensorflow.keras import optimizers
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense,Flatten,Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf

from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# GPU Configuration and Verification
# ============================================================================
print("=" * 60)
print("GPU CONFIGURATION AND SETUP")
print("=" * 60)

# Check TensorFlow version
print(f"TensorFlow version: {tf.__version__}")

# List all available physical devices
print("\nAvailable devices:")
for device in tf.config.list_physical_devices():
    print(f"  - {device}")

# Check for GPU availability
gpus = tf.config.list_physical_devices('GPU')
print(f"\nGPU devices found: {len(gpus)}")

if len(gpus) > 0:
    print("\n✅ GPU DETECTED - Configuring for GPU training...")
    
    # Configure GPU memory growth (prevents TensorFlow from allocating all GPU memory at once)
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
            print(f"  ✓ Memory growth enabled for: {gpu.name}")
        
        # Optional: Set GPU to use mixed precision for faster training (RTX 4080 supports this)
        # Uncomment the following lines if you want to use mixed precision
        # tf.keras.mixed_precision.set_global_policy('mixed_float16')
        # print("  ✓ Mixed precision enabled (float16) - faster training on RTX 4080")
        
        # Get GPU details
        print("\nGPU Details:")
        for i, gpu in enumerate(gpus):
            print(f"  GPU {i}: {gpu.name}")
            # Get GPU memory info
            try:
                gpu_details = tf.config.experimental.get_device_details(gpu)
                if 'device_name' in gpu_details:
                    print(f"    Device: {gpu_details['device_name']}")
                if 'compute_capability' in gpu_details:
                    print(f"    Compute Capability: {gpu_details['compute_capability']}")
            except:
                pass
        
        # Verify GPU is being used with a test computation
        print("\nTesting GPU with a simple operation...")
        with tf.device('/GPU:0'):
            test_tensor = tf.constant([[1.0, 2.0], [3.0, 4.0]])
            result = tf.matmul(test_tensor, test_tensor)
            print(f"  ✓ GPU computation successful!")
            print(f"  ✓ Result device: {result.device}")
            print(f"  ✓ Result: {result.numpy()}")
        
        # Set default device to GPU for all operations
        print("\n  ✓ GPU will be used for all TensorFlow operations")
        
        print("\n" + "=" * 60)
        print("✅ READY FOR GPU TRAINING")
        print("=" * 60)
        print("\n💡 Tips:")
        print("  - Monitor GPU usage: Run 'nvidia-smi -l 1' in terminal")
        print("  - If you get OOM errors, reduce BATCH_SIZE")
        print("  - GPU training will be 10-50x faster than CPU")
        
    except Exception as e:
        print(f"\n⚠️  Warning: Could not configure GPU: {e}")
        print("  Training will use CPU (slower)")
        import traceback
        traceback.print_exc()
else:
    print("\n⚠️  NO GPU DETECTED")
    print("  Training will use CPU (this will be much slower)")
    print("\n  Troubleshooting steps:")
    print("  1. Make sure you RESTARTED the kernel after Cell 0")
    print("  2. Verify NVIDIA driver: Run 'nvidia-smi' in terminal")
    print("  3. For WSL2: Ensure GPU passthrough is enabled")
    print("  4. Try: pip install tensorflow[and-cuda] --upgrade")
    print("  5. Check CUDA libraries: pip list | grep nvidia")
    print("\n" + "=" * 60)
    print("⚠️  USING CPU - Training will be slower")
    print("=" * 60)

# Force GPU usage if available (optional - TensorFlow should auto-select GPU)
if len(gpus) > 0:
    print("\n🔧 Setting default device strategy...")
    # This ensures operations default to GPU
    tf.config.set_visible_devices(gpus[0], 'GPU')
    print("  ✓ GPU set as default device")

print()  # Empty line for readability

GPU CONFIGURATION AND SETUP
TensorFlow version: 2.20.0

Available devices:
  - PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')
  - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

GPU devices found: 1

✅ GPU DETECTED - Configuring for GPU training...
  ✓ Memory growth enabled for: /physical_device:GPU:0

GPU Details:
  GPU 0: /physical_device:GPU:0
    Device: NVIDIA GeForce RTX 4080
    Compute Capability: (8, 9)

Testing GPU with a simple operation...
  ✓ GPU computation successful!
  ✓ Result device: /job:localhost/replica:0/task:0/device:GPU:0
  ✓ Result: [[ 7. 10.]
 [15. 22.]]

  ✓ GPU will be used for all TensorFlow operations

✅ READY FOR GPU TRAINING

💡 Tips:
  - Monitor GPU usage: Run 'nvidia-smi -l 1' in terminal
  - If you get OOM errors, reduce BATCH_SIZE
  - GPU training will be 10-50x faster than CPU

🔧 Setting default device strategy...
  ✓ GPU set as default device



I0000 00:00:1765295125.662714   56430 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13510 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4080, pci bus id: 0000:01:00.0, compute capability: 8.9


# Custom CNN Model for COVID-19 Image Classification

## Model Outline

This notebook implements a **custom CNN model from scratch** for COVID-19 chest X-ray classification. This model will serve as a **baseline for comparison** with pre-trained models from Hugging Face (which will use RAG later).

**Goal**: Create and train a custom model, then compare its performance with external models.

### Overall Structure:
1. **Configuration & Setup** - Define paths, hyperparameters, and constants
2. **Data Loading** - Load train/val splits and create data generators
3. **Data Preprocessing** - Image normalization, resizing, and augmentation
4. **Model Architecture** - Custom CNN design (see tips below)
5. **Model Compilation** - Loss function, optimizer, and metrics
6. **Training** - Training loop with callbacks
7. **Evaluation** - Metrics calculation and visualization
8. **Model Saving** - Save model weights and architecture


## 1. Configuration & Setup


### GPU Installation & Usage Tips:

**✅ Correct Installation for Python 3.10.15:**
- **DO NOT** use `tensorflow-gpu` (deprecated)
- **USE**: Regular `tensorflow>=2.13.0` (works with Python 3.10)
- TensorFlow will automatically detect GPU if CUDA/cuDNN are installed
- If `tensorflow[and-cuda]` fails, use regular `tensorflow` and install CUDA separately
- After installation, **restart the kernel** for GPU to be detected

**To verify GPU is being used during training:**
1. Check the output above - it should show "✅ READY FOR GPU TRAINING"
2. During training, you can monitor GPU usage with:
   - **Windows**: Task Manager → Performance → GPU
   - **nvidia-smi** (if installed): Open terminal and run `nvidia-smi -l 1` to monitor in real-time
3. GPU training will be **10-50x faster** than CPU

**If GPU is not detected after installation:**
1. **Restart the kernel** (Kernel → Restart Kernel in Jupyter)
2. Make sure you have an NVIDIA GPU (AMD GPUs need different setup - ROCm)
3. Verify installation: Run `python -c "import tensorflow as tf; print(tf.config.list_physical_devices('GPU'))"`
4. **For Python 3.10.15**: If GPU isn't detected, you may need to install CUDA/cuDNN separately:
   - Install CUDA Toolkit 11.8 or 12.x (compatible with TensorFlow 2.13+)
   - Install cuDNN 8.x (matching your CUDA version)
   - Download from NVIDIA website

**GPU Memory:**
- The code uses `memory_growth=True` to prevent TensorFlow from allocating all GPU memory
- If you get "out of memory" errors, reduce `BATCH_SIZE` (try 16 or 8)
- You can also reduce `IMG_SIZE` if needed (though this affects model accuracy)

**For Python 3.10.15 specifically:**
- Use `tensorflow>=2.13.0` (compatible with Python 3.10)
- If you get dependency conflicts, try: `pip install tensorflow==2.13.0` (specific version)
- GPU will work if CUDA/cuDNN are properly installed on your system


In [ ]:
# Configuration and hyperparameters
from pathlib import Path
import os

# Data paths
DATA_DIR = Path('data')
TRAIN_SPLIT_CSV = DATA_DIR / 'train_split.csv'
VAL_SPLIT_CSV = DATA_DIR / 'val_split.csv'
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

# Model hyperparameters
IMG_SIZE = 256  # Images are already 256px
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
NUM_CLASSES = 4  # Typical, Negative, Atypical, Indeterminate

# Class labels mapping
CLASS_LABELS = {
    0: 'Negative for Pneumonia',
    1: 'Typical Appearance',
    2: 'Indeterminate Appearance',
    3: 'Atypical Appearance'
}

# Model save paths
MODEL_WEIGHTS_PATH = MODEL_DIR / 'custom_covid19_model.weights.h5'
MODEL_ARCHITECTURE_PATH = MODEL_DIR / 'custom_covid19_model.json'
FEATURES_DIR = MODEL_DIR / 'features'
FEATURES_DIR.mkdir(exist_ok=True)

print("Configuration loaded:")
print(f"  Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Number of classes: {NUM_CLASSES}")


Configuration loaded:
  Image size: 256x256
  Batch size: 32
  Epochs: 50
  Learning rate: 0.001
  Number of classes: 4


## 2. Data Loading & Preprocessing


In [ ]:
# Load train/val splits
train_df = pd.read_csv(TRAIN_SPLIT_CSV)
val_df = pd.read_csv(VAL_SPLIT_CSV)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"\nClass distribution in training set:")
print(train_df['label_id'].value_counts())


Training samples: 5067
Validation samples: 1267

Class distribution in training set:
label_id
Typical Appearance          2406
Negative for Pneumonia      1389
Indeterminate Appearance     886
Atypical Appearance          386
Name: count, dtype: int64


In [5]:
train_df.head()

,ImageInstanceUID,StudyInstanceUID,label_id,study_label,height,width,boxes,image_label,image_path,image_exists
0,000a312787f2,5776db0cec75,Typical Appearance,1,3488,4256,"[{'x': 789.28836, 'y': 582.43035, 'width': 102...",opacity 1 789.28836 582.43035 1815.94498 2499....,data\256px\train\train\5776db0cec75_81456c9c54...,True
1,000c3a3f293f,ff0879eb20ed,Negative for Pneumonia,0,2320,2832,"[{'x': 0, 'y': 0, 'width': 1, 'height': 1}]",none 1 0 0 1 1,data\256px\train\train\ff0879eb20ed_d8a644cc4f...,True
2,001398f4ff4f,28dddc8559b2,Atypical Appearance,3,3520,4280,"[{'x': 2729, 'y': 2181.33331, 'width': 948.000...",opacity 1 2729 2181.33331 3677.00012 2785.33331,data\256px\train\train\28dddc8559b2_4d47bc042e...,True
3,001bd15d1891,dfd9fdd85a3e,Typical Appearance,1,2800,3408,"[{'x': 623.23328, 'y': 1050, 'width': 714, 'he...",opacity 1 623.23328 1050 1337.23328 2156 opaci...,data\256px\train\train\dfd9fdd85a3e_49170afa4f...,True
4,0022227f5adf,84543edc24c2,Indeterminate Appearance,2,2539,3050,"[{'x': 1857.2065, 'y': 508.30565, 'width': 376...",opacity 1 1857.2065 508.30565 2233.23384 907.8...,data\256px\train\train\84543edc24c2_82f65ab98e...,True


## 3. Data Preprocessing & Augmentation


In [6]:
# Custom data generator for loading images
def load_and_preprocess_image(image_path, target_size=(IMG_SIZE, IMG_SIZE)):
    """Load and preprocess a single image"""
    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(f"Could not load image: {image_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, target_size)
    img = img.astype('float32') / 255.0  # Normalize to [0, 1]
    return img

# Data augmentation for training
train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

# No augmentation for validation
val_datagen = ImageDataGenerator()

print("Data generators created with augmentation for training set")


Data generators created with augmentation for training set


## 4. Model Architecture - Custom CNN


In [7]:
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, BatchNormalization, 
    Dropout, GlobalAveragePooling2D, Activation
)

def create_custom_cnn_model(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    """
    Create a custom CNN model for COVID-19 image classification.
    The model includes:
    - Multiple convolutional blocks with batch normalization
    - Dropout for regularization
    - Global average pooling to reduce parameters
    - Feature extraction layer (for RAG integration later)
    - Classification head
    """
    model = Sequential([
        Input(shape=input_shape),
        
        # First convolutional block
        Conv2D(32, (3, 3), padding='same'),
        BatchNormalization(),
        Activation('relu'),
        Conv2D(32, (3, 3), padding='same'),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        # Second convolutional block
        Conv2D(64, (3, 3), padding='same'),
        BatchNormalization(),
        Activation('relu'),
        Conv2D(64, (3, 3), padding='same'),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        # Third convolutional block
        Conv2D(128, (3, 3), padding='same'),
        BatchNormalization(),
        Activation('relu'),
        Conv2D(128, (3, 3), padding='same'),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        # Fourth convolutional block
        Conv2D(256, (3, 3), padding='same'),
        BatchNormalization(),
        Activation('relu'),
        Conv2D(256, (3, 3), padding='same'),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        # Feature extraction layer (for RAG - this will be used to extract embeddings)
        GlobalAveragePooling2D(),
        Dense(512, activation='relu', name='feature_extraction'),
        BatchNormalization(),
        Dropout(0.5),
        
        # Classification head
        Dense(num_classes, activation='softmax', name='classification')
    ])
    
    return model

# Create the model
model = create_custom_cnn_model()
model.summary()



Model Architecture:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 256, 256, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256, 256, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 256, 256, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 256, 256, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256, 256, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 256, 256, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 128, 128, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128, 128, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 128, 128, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 128, 128, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128, 128, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 128, 128, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 64, 64, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 64, 64, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 64, 64, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 64, 64, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 64, 64, 128)    │             

 Total params: 1,311,780 (5.00 MB)

 Trainable params: 1,308,836 (4.99 MB)

 Non-trainable params: 2,944 (11.50 KB)


Total trainable parameters: 1,311,780
Model size: ~5.00 MB (float32)


## 5. Model Compilation


In [8]:
# Compile the model
model.compile(
    optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=losses.SparseCategoricalCrossentropy(),  # Use sparse since labels are integers
    metrics=['accuracy']
)

print("Model compiled successfully!")
print(f"  Optimizer: Adam (lr={LEARNING_RATE})")
print(f"  Loss: Sparse Categorical Crossentropy")
print(f"  Metrics: Accuracy")


Class weights (to handle imbalanced data):
  Class 0 (Negative for Pneumonia): 0.912
  Class 1 (Typical Appearance): 0.526
  Class 2 (Indeterminate Appearance): 1.430
  Class 3 (Atypical Appearance): 3.282

Model compiled successfully!
  Optimizer: Adam (lr=0.001)
  Loss: Sparse Categorical Crossentropy
  Metrics: Accuracy
  Class weights: Will be used during training to handle imbalance


## 6. Training Setup - Callbacks


In [12]:
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, 
    CSVLogger, TensorBoard
)

# Define callbacks
callbacks = [
    # Save best model based on validation accuracy
    ModelCheckpoint(
        filepath=str(MODEL_WEIGHTS_PATH),
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=True,
        mode='max',
        verbose=1
    ),
    
    # Early stopping to prevent overfitting
    EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Reduce learning rate when validation loss plateaus
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    
    # Log training history
    CSVLogger(
        filename=str(MODEL_DIR / 'training_history.csv'),
        append=False
    ),
    
    # TensorBoard logging (optional - for visualization)
    TensorBoard(
        log_dir=str(MODEL_DIR / 'logs'),
        histogram_freq=1
    )
]

print("Callbacks configured:")
print("  - ModelCheckpoint: Save best model weights")
print("  - EarlyStopping: Stop if no improvement for 10 epochs")
print("  - ReduceLROnPlateau: Reduce LR when validation plateaus")
print("  - CSVLogger: Save training history")
print("  - TensorBoard: Log training metrics")


Callbacks configured:
  - ModelCheckpoint: Save best model weights
  - EarlyStopping: Stop if no improvement for 10 epochs
  - ReduceLROnPlateau: Reduce LR when validation plateaus
  - CSVLogger: Save training history
  - TensorBoard: Log training metrics


## Step 7 Explained: Data Generators for Training

### Why Custom Data Generators?

**Problem**: Loading all images into memory at once would:
- Require ~6GB+ RAM (5067 images × 256×256×3 × 4 bytes)
- Be slow to start training
- Make data augmentation inefficient

**Solution**: Custom data generator that loads images **on-demand** (lazy loading):
- Only loads one batch at a time into memory
- Efficient memory usage (~100MB per batch)
- Enables real-time data augmentation
- Works seamlessly with Keras `model.fit()`

### How It Works:

The `CustomDataGenerator` class inherits from `tf.keras.utils.Sequence`, which:
- Makes it compatible with Keras training
- Ensures proper multi-processing support
- Handles epoch boundaries correctly


## 7. Data Generators for Training


In [ ]:
# Create custom data generator class
class CustomDataGenerator(tf.keras.utils.Sequence):
    """Custom data generator for loading images and labels"""
    
    def __init__(self, dataframe, batch_size=BATCH_SIZE, shuffle=True, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment
        self.on_epoch_end()
    
    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))
    
    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_df = self.df.iloc[batch_indices]
        
        X = np.zeros((len(batch_df), IMG_SIZE, IMG_SIZE, 3))
        y = np.zeros(len(batch_df))
        
        for i, (_, row) in enumerate(batch_df.iterrows()):
            # Load and preprocess image
            img = load_and_preprocess_image(row['image_path'])
            
            # Apply augmentation if training
            if self.augment:
                img = train_datagen.random_transform(img)
            
            X[i] = img
            y[i] = row['study_label']  # Use numeric label
        
        return X, y
    
    def on_epoch_end(self):
        self.indices = np.arange(len(self.df))
        if self.shuffle:
            np.random.shuffle(self.indices)

# Create data generators
train_generator = CustomDataGenerator(
    train_df, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    augment=True
)

val_generator = CustomDataGenerator(
    val_df, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    augment=False
)

print(f"Training batches: {len(train_generator)}")
print(f"Validation batches: {len(val_generator)}")


Training batches: 159
Validation batches: 40


## 8. Model Training


In [ ]:
# Train the model
print("=" * 60)
print("MODEL TRAINING")
print("=" * 60)

# Verify GPU is available before training
gpus = tf.config.list_physical_devices('GPU')
if len(gpus) > 0:
    print(f"\n✅ GPU detected: {len(gpus)} GPU(s) available")
    print(f"   Using GPU: {gpus[0].name}")
    print("   Training will use GPU acceleration (10-50x faster than CPU)")
    print("\n💡 Monitor GPU usage: Run 'nvidia-smi -l 1' in terminal")
else:
    print("\n⚠️  WARNING: No GPU detected!")
    print("   Training will use CPU (much slower)")
    print("   Consider restarting kernel and checking GPU setup in Cell 3")

print("\n" + "=" * 60)
print("Starting training...")
print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print("=" * 60 + "\n")

# Train the model
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print("\n" + "=" * 60)
print("✅ Training completed!")
print("=" * 60)


Starting training...
Training on 5067 samples
Validating on 1267 samples
Using class weights to handle imbalanced data
Epoch 1/50
 22/159 ━━━━━━━━━━━━━━━━━━━━ 5:13 2s/step - accuracy: 0.2666 - loss: 1.8292

## 9. Training History Visualization


In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Training Accuracy', marker='o')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', marker='s')
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='Training Loss', marker='o')
axes[1].plot(history.history['val_loss'], label='Validation Loss', marker='s')
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final metrics
print("\nFinal Training Metrics:")
print(f"  Training Accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"  Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")
print(f"  Training Loss: {history.history['loss'][-1]:.4f}")
print(f"  Validation Loss: {history.history['val_loss'][-1]:.4f}")


## 10. Model Evaluation - Detailed Metrics


In [ ]:
# Load best model weights
model.load_weights(str(MODEL_WEIGHTS_PATH))

# Predict on validation set
print("Evaluating on validation set...")
val_predictions = model.predict(val_generator, verbose=1)
val_predicted_classes = np.argmax(val_predictions, axis=1)
val_true_classes = val_df['study_label'].values

# Calculate metrics
accuracy = accuracy_score(val_true_classes, val_predicted_classes)
precision = precision_score(val_true_classes, val_predicted_classes, average='weighted')
recall = recall_score(val_true_classes, val_predicted_classes, average='weighted')
f1 = f1_score(val_true_classes, val_predicted_classes, average='weighted')

print("\nValidation Set Metrics:")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall: {recall:.4f}")
print(f"  F1-Score: {f1:.4f}")

# Confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(val_true_classes, val_predicted_classes)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[CLASS_LABELS[i] for i in range(NUM_CLASSES)],
            yticklabels=[CLASS_LABELS[i] for i in range(NUM_CLASSES)])
plt.title('Confusion Matrix - Validation Set', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Classification report
print("\nClassification Report:")
print(classification_report(val_true_classes, val_predicted_classes, 
                          target_names=[CLASS_LABELS[i] for i in range(NUM_CLASSES)]))


## 11. Save Model Architecture


In [ ]:
# Save model architecture as JSON
model_json = model.to_json()
with open(MODEL_ARCHITECTURE_PATH, 'w') as json_file:
    json_file.write(model_json)

print(f"Model architecture saved to: {MODEL_ARCHITECTURE_PATH}")
print(f"Model weights saved to: {MODEL_WEIGHTS_PATH}")


## 12. Feature Extraction for RAG Integration

This section extracts feature embeddings from the model's feature extraction layer. These embeddings will be useful when integrating with external models and RAG systems.


In [ ]:
# Create a model that outputs features from the feature extraction layer
feature_extractor = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer('feature_extraction').output
)

# Extract features for all training images (for RAG vector database)
print("Extracting features from training set...")
train_features = []
train_labels = []

for i in range(len(train_generator)):
    X_batch, y_batch = train_generator[i]
    features_batch = feature_extractor.predict(X_batch, verbose=0)
    train_features.append(features_batch)
    train_labels.append(y_batch)

train_features = np.vstack(train_features)
train_labels = np.hstack(train_labels)

print(f"Training features shape: {train_features.shape}")
print(f"Training labels shape: {train_labels.shape}")

# Extract features for validation set
print("\nExtracting features from validation set...")
val_features = []
val_labels = []

for i in range(len(val_generator)):
    X_batch, y_batch = val_generator[i]
    features_batch = feature_extractor.predict(X_batch, verbose=0)
    val_features.append(features_batch)
    val_labels.append(y_batch)

val_features = np.vstack(val_features)
val_labels = np.hstack(val_labels)

print(f"Validation features shape: {val_features.shape}")
print(f"Validation labels shape: {val_labels.shape}")

# Save features for later use with RAG
np.save(FEATURES_DIR / 'train_features.npy', train_features)
np.save(FEATURES_DIR / 'train_labels.npy', train_labels)
np.save(FEATURES_DIR / 'val_features.npy', val_features)
np.save(FEATURES_DIR / 'val_labels.npy', val_labels)

print(f"\nFeatures saved to: {FEATURES_DIR}")
print("These features can be used for RAG integration with external models.")


## Summary

### Model Architecture:
- **Input**: 256x256x3 RGB images
- **Architecture**: Custom CNN with 4 convolutional blocks
- **Feature Extraction**: 512-dimensional embeddings (for RAG)
- **Output**: 4-class classification (Typical, Negative, Atypical, Indeterminate)

### Next Steps for RAG Integration:
1. Use extracted features to build a vector database
2. Integrate external pre-trained models (e.g., ResNet, EfficientNet, Vision Transformers)
3. Combine custom model features with external model features
4. Implement RAG pipeline for retrieval-augmented classification
5. Compare performance between custom model and RAG-enhanced model


## Additional Tips for Experimentation

### Hyperparameter Tuning Strategy:
1. **Start with this baseline** - Train and evaluate current architecture
2. **Monitor overfitting** - If train acc >> val acc, increase dropout or reduce model size
3. **Monitor underfitting** - If both accuracies are low, increase model capacity or train longer
4. **Learning rate** - If loss doesn't decrease, try lower LR (0.0001) or use learning rate finder
5. **Batch size** - Larger batches (64) can be more stable, smaller (16) can help generalization

### Architecture Experiments to Try:
- **Residual blocks**: Add skip connections for deeper networks
- **Depthwise separable convolutions**: Reduce parameters while maintaining capacity
- **Attention layers**: Help model focus on lung regions and opacities
- **Multi-scale features**: Combine features from different depths (like FPN)

### Data Augmentation Tweaks:
- **Medical imaging specific**: 
  - Be careful with horizontal flips (chest X-rays are usually oriented correctly)
  - Consider contrast adjustment (simulate different X-ray machines)
  - Brightness/contrast augmentation can help with varying image quality

### Evaluation Focus:
- **Per-class metrics**: Pay attention to minority classes (Atypical, Indeterminate)
- **Confusion matrix**: Identify which classes are confused most
- **Visual inspection**: Look at misclassified images to understand model limitations
